In [ ]:
from google.colab import files


print("Upload your full snapshot image:")
uploaded = files.upload()
input_path = list(uploaded.keys())[0]

Upload your full snapshot image:


Saving 0.png to 0.png


In [ ]:
# Setup environment
!pip install -q diffusers transformers accelerate safetensors pillow torch torchvision xformers


import torch
from diffusers import DiffusionPipeline
from PIL import Image


device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

input_image = Image.open(input_path).convert("RGB")

# load both base & refiner
base = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True
)
refiner = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    text_encoder_2=base.text_encoder_2,
    vae=base.vae,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
)
refiner.to(device)
refiner.enable_model_cpu_offload()
refiner.enable_xformers_memory_efficient_attention()
# # remove the line if xFormers is not installed or you have PyTorch 2.0 or higher installed

prompt = "16th century Age of Discovery coastline of western Taiwan, sunset, sandbars and shallow coastline, mountain in the back, Portuguese ships anchoring in distance, warm tropical climate, cinematic realism, detailed architecture"
n_steps = 40
high_noise_frac = 0.8

output_image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=input_image,
    height=768,
    width=768,
).images[0]
output_image.save("historical_result.png")
print('Complete')


Using device: cuda


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]